# 05 · Comparar modelos y geometrías — solución de referencia

Comparar modelos con el mismo protocolo y contrastar k-means, dbscan, jerárquico y espectral sobre dos geometrías.

**Ideas que aparecen:** laboratorios 02 y 04; distancia entre vectores. Puedes consultar las cápsulas
opcionales de `ruta/Puentes de entrada.md` cuando alguna te haga falta.

Datos **sintéticos educativos**, creados en este repositorio y dedicados a
CC0-1.0. No representan personas ni un rendimiento oficial IOAI.
Todo el ejercicio usa CPU y archivos locales; no requiere cuentas ni red.

Esta es una propuesta para comparar con tus ideas; no es la única solución posible.
Puedes recorrerlo en una o varias sesiones. No hay límite de juez ni
obligación de completar todos los experimentos para abrir el siguiente cuaderno.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})

DATOS = Path("datos")
assert DATOS.is_dir(), "Abre el notebook desde su carpeta: deben existir datos/ e inicio.ipynb."
SEMILLA = 17


## Comparar necesita una pregunta fija
Primero clasifica puntos con frontera curva. Fija modelos y
parámetros antes de medir; elige por validación y congela la
elección antes de transferencia. El baseline mayoritario se mide
sobre las mismas filas.

Después elimina la supervisión: ajusta agrupadores usando solo las
coordenadas. K-means busca grupos compactos alrededor de centros;
DBSCAN une regiones densas y puede dejar ruido con etiqueta -1;
jerárquico construye un árbol de fusiones (aquí enlace simple);
espectral usa la conectividad de un grafo de vecindad.

Como los puntos son sintéticos conocemos su grupo generador y
calculamos ARI (índice de Rand ajustado) **después** del ajuste:
1 indica coincidencia, cerca de 0 acuerdo esperado por azar y puede
ser negativo. Los números asignados a los grupos son arbitrarios:
accuracy directa sería incorrecta. En datos sin verdad de referencia
no puedes calcular ARI; examina geometría, estabilidad y objetivo.
No optimices con estas etiquetas ocultadas durante el ajuste.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, adjusted_rand_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering, SpectralClustering
df = pd.read_csv(DATOS / "clasificacion.csv")
columnas = ["x1", "x2"]
train, val, nuevo = [df[df.particion == p] for p in ["train", "val", "transferencia"]]


## Zona de experimentación
Agrega logística, k-NN (k=7), árbol (profundidad 4) y SVM RBF
(C=2) al diccionario; escala dentro del pipeline cuando corresponda.
Puedes modificar esos parámetros después para ver qué geometría cambia. La comparación es
local, no un ranking universal de algoritmos.

Para clustering implementa `agrupar`: KMeans (k=2, n_init=10),
DBSCAN (eps=0.22, min_samples=5), jerárquico (2 grupos, enlace
simple) y espectral (2 grupos, 12 vecinos, afinidad nearest_neighbors).
Predice cuáles pueden recuperar dos medias lunas antes de medir.
Estas coordenadas ya comparten unidad; no las reescales por separado.


In [ ]:
modelos = {
    "mayoritario": DummyClassifier(strategy="most_frequent"),
    "logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEMILLA)),
    "knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "arbol": DecisionTreeClassifier(max_depth=4, random_state=SEMILLA),
    "svm": make_pipeline(StandardScaler(), SVC(C=2, kernel="rbf")),
}

def agrupar(X):
    algoritmos = {
        "kmeans": KMeans(n_clusters=2, n_init=10, random_state=SEMILLA),
        "dbscan": DBSCAN(eps=0.22, min_samples=5),
        "jerarquico": AgglomerativeClustering(n_clusters=2, linkage="single"),
        "espectral": SpectralClustering(n_clusters=2, n_neighbors=12,
                                        affinity="nearest_neighbors", random_state=SEMILLA),
    }
    return {nombre: algoritmo.fit_predict(X) for nombre, algoritmo in algoritmos.items()}


## Comparación en las mismas condiciones
Elige mayor accuracy de validación; desempata por nombre en orden alfabético. Reporta también train para detectar diferencias de ajuste. Para clustering compara ARI con un solo grupo (0 en estos datos) y reporta el ruido en lugar de eliminarlo silenciosamente.


In [ ]:
tabla = []
for nombre, modelo in modelos.items():
    modelo.fit(train[columnas], train.clase)
    tabla.append({"modelo": nombre,
                  "train": float(accuracy_score(train.clase, modelo.predict(train[columnas]))),
                  "val": float(accuracy_score(val.clase, modelo.predict(val[columnas])))})
tabla = pd.DataFrame(tabla).sort_values(["val", "modelo"], ascending=[False, True])
elegido = str(tabla.iloc[0].modelo)
resultado = {"baseline": float(tabla.set_index("modelo").loc["mayoritario", "val"]),
             "validacion": float(tabla.iloc[0].val), "elegido": elegido,
             "comparacion": tabla.to_dict(orient="records")}
print(tabla)

def comparar_clusters(archivo):
    puntos = pd.read_csv(DATOS / archivo)
    # La columna grupo jamás entra a agrupar: solo x1 y x2.
    asignaciones = agrupar(puntos[columnas].to_numpy())
    medidas = {}
    for nombre, etiquetas in asignaciones.items():
        assert len(etiquetas) == len(puntos)
        medidas[nombre] = {"ari": float(adjusted_rand_score(puntos.grupo, etiquetas)),
                           "ruido": float(np.mean(etiquetas == -1)),
                           "grupos": len(set(etiquetas) - {-1})}
    return medidas

resultado["clusters_lunas"] = comparar_clusters("lunas.csv")
print("Medias lunas:", resultado["clusters_lunas"])


<details><summary>Idea · comparación</summary>Usa los mismos train y val para cada modelo; el escalador va dentro de cada pipeline.</details>
<details><summary>Idea · agrupación</summary>Cada clase de sklearn.cluster ofrece fit_predict(X); los nombres de etiquetas no tienen significado ordinal.</details>
<details><summary>Idea · ruido y conexión</summary>DBSCAN puede devolver -1. Si un grafo espectral no está conectado aparece un aviso: revisa su geometría y no lo ocultes como si fuera un entrenamiento fallido.</details>


## Transferencia: decide antes de mirar el resultado
Evalúa el clasificador ya elegido en 80 filas nuevas. Luego conserva los cuatro agrupadores y sus parámetros sobre dos nubes compactas. Esta segunda parte vuelve a ajustar agrupaciones sobre otro conjunto completo: es una transferencia del procedimiento, no predict de un clusterizador sobre puntos nuevos. Compara cómo cambia el resultado con la geometría.


In [ ]:
resultado["transferencia"] = float(accuracy_score(nuevo.clase, modelos[elegido].predict(nuevo[columnas])))
resultado["baseline_transferencia"] = float(accuracy_score(nuevo.clase, modelos["mayoritario"].predict(nuevo[columnas])))
resultado["clusters_nubes"] = comparar_clusters("nubes.csv")
# Renombrar clusters no debe cambiar ARI; una accuracy directa sí cambiaría.
grupos = np.array([0, 0, 1, 1])
assert adjusted_rand_score(grupos, 1 - grupos) == 1.0


## Mirar la idea
Compara el dibujo con lo que esperabas antes de ejecutar.


In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.6, 1.6, 90), np.linspace(-1.6, 1.6, 90))
grilla = pd.DataFrame({"x1": xx.ravel(), "x2": yy.ravel()})
fig, ejes = plt.subplots(1, len(modelos), figsize=(3.1 * len(modelos), 3.3), squeeze=False)
for eje, (nombre, modelo) in zip(ejes.ravel(), modelos.items()):
    eje.contourf(xx, yy, modelo.predict(grilla).reshape(xx.shape), levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.25)
    eje.scatter(val.x1, val.x2, c=val.clase, cmap="coolwarm", s=16, edgecolor="white", linewidth=0.3)
    eje.set(title=nombre, xlabel="x1", ylabel="x2", aspect="equal")
fig.suptitle("Cada clasificador propone una geometría de frontera", y=1.04)
fig.tight_layout()
plt.show()

for archivo, titulo in [("lunas.csv", "Medias lunas"), ("nubes.csv", "Nubes compactas")]:
    puntos = pd.read_csv(DATOS / archivo)
    asignaciones = agrupar(puntos[columnas].to_numpy())
    fig, ejes = plt.subplots(1, len(asignaciones), figsize=(3.3 * len(asignaciones), 3.3), squeeze=False)
    for eje, (nombre, etiquetas) in zip(ejes.ravel(), asignaciones.items()):
        ruido = etiquetas == -1
        eje.scatter(puntos.loc[~ruido, "x1"], puntos.loc[~ruido, "x2"], c=etiquetas[~ruido], cmap="coolwarm", s=18)
        eje.scatter(puntos.loc[ruido, "x1"], puntos.loc[ruido, "x2"], color="black", marker="x", label="ruido")
        ari = adjusted_rand_score(puntos.grupo, etiquetas)
        eje.set(title=f"{nombre}; ARI={ari:.2f}", xlabel="x1", ylabel="x2", aspect="equal")
    fig.suptitle(titulo + ": distintas ideas de parecido", y=1.04)
    fig.tight_layout()
    plt.show()


## Para seguir explorando
¿Qué forma de frontera aprende cada clasificador? Compara medias lunas y nubes compactas: ¿qué noción de grupo encaja en cada dibujo? Cambia eps de DBSCAN y observa cuándo aparecen ruido, uniones o fragmentos; no hay un ganador universal.


## Comprobaciones del ejemplo
Estas aserciones detectan errores técnicos en el cuaderno y en su solución de referencia. No son un examen ni una escala de capacidad.


In [ ]:
assert len(resultado['comparacion']) == 5
assert resultado['validacion'] >= 0.85 and resultado['transferencia'] >= 0.80
assert set(resultado['clusters_lunas']) == {'kmeans', 'dbscan', 'jerarquico', 'espectral'}
assert set(resultado['clusters_nubes']) == set(resultado['clusters_lunas'])
assert resultado['clusters_lunas']['dbscan']['ari'] > resultado['clusters_lunas']['kmeans']['ari'] + 0.2
assert resultado['clusters_nubes']['kmeans']['ari'] > 0.9


## Resultado reproducible
Esta celda guarda automáticamente las medidas para comprobar el material. Puedes conservar una copia de tu notebook y tus propias notas; no hay un formulario que rellenar.


In [ ]:
resultado.update({"laboratorio": '05_modelos', "version": 'solución de referencia',
                  "datos": "sintéticos CC0-1.0",
                  "metrica": 'accuracy supervisada; ARI educativo en clustering, fracción de ruido y número de grupos', "split": 'clasificación: 160 train/80 validación/80 transferencia; clustering: dos conjuntos independientes de 160 puntos'})
Path("resultado.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=2))
